In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import (cosine_similarity, euclidean_distances, manhattan_distances)

load_dotenv()  # Load environment variables from .env file

client = OpenAI(api_key=os.environ.get("openrouterapikey"), base_url="https://openrouter.ai/api/v1")


EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"
print("Embedding model:", EMBEDDING_MODEL)

Embedding model: nvidia/nemotron-3-embed-1b:free


In [14]:
documetns = [
    {
        "id": "1",
        "text": "Python is a popular programming language known for its simple syntax.",
        "topic": "Python",
        "level": "Beginner"
    },
    {
        "id": "2",
        "text": "Python lists are used to store multiple items in a single variable.",
        "topic": "Python",
        "level": "Beginner"
    },
    {
        "id": "3",
        "text": "Python dictionaries store data as key-value pairs.",
        "topic": "Python",
        "level": "Beginner"
    },
    {
        "id": "4",
        "text": "Python functions allow you to reuse code and organize logic into smaller blocks.",
        "topic": "Python",
        "level": "Intermediate"
    },
    {
        "id": "5",
        "text": "NumPy arrays support efficient mathematical operations on numerical data.",
        "topic": "Python",
        "level": "Intermediate"
    },
    {
        "id": "6",
        "text": "Generative AI models can create new content such as text, images, and code.",
        "topic": "Generative AI",
        "level": "Beginner"
    },
    {
        "id": "7",
        "text": "Large language models generate text by predicting likely next tokens based on context.",
        "topic": "Generative AI",
        "level": "Beginner"
    },
    {
        "id": "8",
        "text": "Embeddings represent text as numerical vectors that capture semantic information.",
        "topic": "Generative AI",
        "level": "Intermediate"
    },
    {
        "id": "9",
        "text": "Vector databases store and search embeddings to find semantically similar information.",
        "topic": "Generative AI",
        "level": "Intermediate"
    },
    {
        "id": "10",
        "text": "Retrieval-Augmented Generation retrieves relevant documents and provides them as context to an LLM.",
        "topic": "Generative AI",
        "level": "Advanced"
    }
]

texts=[doc["text"] for doc in documetns]
print(f"{len(documetns)} documents to embed")

10 documents to embed


In [22]:
def embed_documents(texts):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    embeddings = [item.embedding for item in response.data]
    return embeddings

doc_embeddings = embed_documents(texts)
print(f"Embeddings generated for {len(doc_embeddings)} documents")
print("dimension of each embedding:", len(doc_embeddings[0]))
print("Sample embedding for first document:", doc_embeddings[0][:5])  # Print first 10 dimensions of the first embedding

Embeddings generated for 10 documents
dimension of each embedding: 2048
Sample embedding for first document: [0.11399596929550171, 0.008722899481654167, -0.02692873775959015, 0.002512727864086628, -0.02889556996524334]


In [29]:
query="Which Programming language is best for AI and commonly used for AI Application?"

query_embedding = embed_documents([query])
print("query:", query)
print("Query embedding generated. Dimension:", len(query_embedding[0]))
print("collection of embeddings generated for documents. Dimension:", len(doc_embeddings[0]))


query: Which Programming language is best for AI and commonly used for AI Application?
Query embedding generated. Dimension: 2048
collection of embeddings generated for documents. Dimension: 2048


In [35]:
cos=cosine_similarity(query_embedding, doc_embeddings)[0]
l2=euclidean_distances(query_embedding, doc_embeddings)[0]
print("Cosine similarity:", cos)
print("Euclidean distances:", l2)
for c,d in zip(cos, l2):
    print(f"Cosine similarity: {c:.4f}, Euclidean distance: {d:.4f}")   

Cosine similarity: [0.28565804 0.10647766 0.05940101 0.0432848  0.08448888 0.20203683
 0.10509041 0.04214829 0.05549407 0.11359756]
Euclidean distances: [1.19527556 1.33680405 1.37156759 1.38326829 1.35315288 1.26329971
 1.33784108 1.3840894  1.37441338 1.33146727]
Cosine similarity: 0.2857, Euclidean distance: 1.1953
Cosine similarity: 0.1065, Euclidean distance: 1.3368
Cosine similarity: 0.0594, Euclidean distance: 1.3716
Cosine similarity: 0.0433, Euclidean distance: 1.3833
Cosine similarity: 0.0845, Euclidean distance: 1.3532
Cosine similarity: 0.2020, Euclidean distance: 1.2633
Cosine similarity: 0.1051, Euclidean distance: 1.3378
Cosine similarity: 0.0421, Euclidean distance: 1.3841
Cosine similarity: 0.0555, Euclidean distance: 1.3744
Cosine similarity: 0.1136, Euclidean distance: 1.3315


In [40]:
def retrieve_top_k_documents(query_embedding, doc_embeddings, documents, k=3):
    cos_sim = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_k_indices = cos_sim.argsort()[-k:][::-1]  # Get indices of top k documents
    top_k_docs = [documents[i] for i in top_k_indices]
    return top_k_docs

print(f"\nTop 3 documents retrieved for the query '{query}':")
top_k_docs = retrieve_top_k_documents(query_embedding, doc_embeddings, documetns, k=3)
for doc in top_k_docs:
    print(f"Document ID: {doc['id']}, Text: {doc['text']}")


Top 3 documents retrieved for the query 'Which Programming language is best for AI and commonly used for AI Application?':
Document ID: 1, Text: Python is a popular programming language known for its simple syntax.
Document ID: 6, Text: Generative AI models can create new content such as text, images, and code.
Document ID: 10, Text: Retrieval-Augmented Generation retrieves relevant documents and provides them as context to an LLM.


Qdrant Vector Database


In [44]:
from qdrant_client import QdrantClient, models
qdrant=QdrantClient(url="http://localhost:6333")  # Connect to Qdrant running locally
#qdrant=QdrantClient(":memory:")  # Connect to Qdrant running in-memory for testing

In [47]:
COLLECTION_NAME = "rag_documents"
# Create a collection in Qdrant if it doesn't exist
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),  # Dimension of the embeddings
        distance=models.Distance.COSINE  # Use cosine distance for similarity search        
    ),
)

print(f"Collection '{COLLECTION_NAME}' created in Qdrant.")
print(f"Uploading {len(doc_embeddings[0])} document embeddings to Qdrant...")

Collection 'rag_documents' created in Qdrant.
Uploading 2048 document embeddings to Qdrant...


In [58]:
import uuid  # Import UUID module for generating valid UUIDs

# Ensure all IDs are valid (unsigned integers or UUIDs)
points = [
    models.PointStruct(
        id=int(doc["id"]) if isinstance(doc["id"], int) and doc["id"] >= 0 else str(uuid.UUID(doc["id"])) if isinstance(doc["id"], str) and len(doc["id"]) == 36 else str(uuid.uuid4()),  # Validate or generate ID
        vector=vec,
        payload={"text": doc["text"], "topic": doc["topic"], "level": doc["level"]}
    ) 
    for doc, vec in zip(documetns, doc_embeddings)
]

qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True  # Wait for the operation to complete
)
print(f"{len(points)} document embeddings uploaded to Qdrant.")

10 document embeddings uploaded to Qdrant.


In [68]:
compressedClient = qdrant.create_collection(
    collection_name='compressed_rag_documents2',
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),  # Dimension of the embeddings
        distance=models.Distance.COSINE  # Use cosine distance for similarity search        
    ),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,  # Correct field name
            always_ram=True  # Correct placement
        )
    )
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `compressed_rag_documents2` already exists!"},"time":0.000332584}'

In [69]:
qdrant.upsert(
    collection_name='compressed_rag_documents2',
    points=points,
    wait=True  # Wait for the operation to complete
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [73]:
results = qdrant.query_points(
    collection_name='compressed_rag_documents2',
    query=query_embedding[0],
    limit=3,  # Retrieve top 3 documents,
    with_payload=True  # Include payload in the results
).points

print(f"\nTop 3 documents retrieved from Qdrant for the query '{query}':")
print(results[0].payload)


Top 3 documents retrieved from Qdrant for the query 'Which Programming language is best for AI and commonly used for AI Application?':
{'text': 'Python is a popular programming language known for its simple syntax.', 'topic': 'Python', 'level': 'Beginner'}
